# Attention over K (x_i, CF) pairs — head fitted on the val split

Replaces the mean of K sigmoids with gated attention pooling (Ilse et al. 2018, Eq. 4 in Shad et al. 2026) over the K pair embeddings.

Per fold:
1. Load the **frozen** K=1 CNN checkpoint (`fold_<i>_best.pt`).
2. Rebuild the val split that chose its epoch (same seed, same `VAL_FRAC`, grouped by patient).
3. Embed every (x_i, CF_k) pair of the **val** and **test** rows.
4. Fit the attention head on val, evaluate on test, compare with the mean of K sigmoids on the same rows.

Nothing but the head is trained.

In [5]:
# ── Config ────────────────────────────────────────────────────────────────────
CODE_DIR    = "/zhome/d0/a/221493/thesis/final_experiment/code"           # folder holding c2_train_images.py / c2_folds.py
DISEASE     = "effusion"
MODEL_KEY   = "dual_sal_scalars"     # dual, dual_sal, dual_sal_scalars, cf, cf_sal
K           = 5
POLICY      = "ignore"       # C0 policy -> Grad-CAM directory

CF_SOURCE   = dict(kind="knn", strategy="correct_cf", distance="l1", k_offset=0)
N_FOLDS     = 5
VAL_FRAC    = 0.15           # MUST match the c2_train_images.py run

EVAL_BATCH  = 32 // K        # pairs in flight = EVAL_BATCH x K
NUM_WORKERS = 4

# head
ATTN_DIM, DROPOUT = 128, 0.25
LR, WEIGHT_DECAY  = 1e-3, 5e-4
EPOCHS, PATIENCE  = 100, 10
HEAD_VAL_FRAC     = 0.2      # slice of val used for the head's early stopping
BATCH             = 256

In [6]:
import os, sys
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath(CODE_DIR))
import c2_train_images as cti
from c2_folds import RESULTS_DIR, RANDOM_SEED, load_folds
from c2_cf_sources import build_cf_source

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
src = build_cf_source(**CF_SOURCE)
cfg = cti.MODEL_REGISTRY[MODEL_KEY]
cti.CAM_BASE = os.path.join(RESULTS_DIR, "C0_final", f"multilabel_{POLICY}", DISEASE, "gradcam")

RUN      = os.path.join(RESULTS_DIR, "C2_final_results", DISEASE, src.name)
CKPT_DIR = os.path.join(RUN, "cf_1", "images", MODEL_KEY)          # frozen K=1 models
OUT_DIR  = os.path.join(RUN, f"cf_{K}", "attention_val", MODEL_KEY)
os.makedirs(OUT_DIR, exist_ok=True)
print(device, "|", src.describe(), "|", CKPT_DIR)

Using device: cuda
cuda | KNN / correct_cf / distance=l1 | /work3/s251710/thesis_results/C2_final_results/effusion/knn_correct_cf/cf_1/images/dual_sal_scalars


## Folds
Same folds and pairing as the K sweep. Each row carries its K nearest CFs in `cf_paths`.

In [7]:
prob_col = f"{DISEASE}_prob"
folds = list(load_folds(DISEASE, src, K, n_folds=N_FOLDS, materialize=False,
                        columns=cti.FOLD_COLS + [prob_col]))
if cfg["mode"] == "quad_scalars":
    cti._attach_scalar_cols([d for _, tr, te in folds for d in (tr, te)], DISEASE, prob_col)

def val_split(train_df):
    """The exact train_sub / val split c2_train_images.run_cv used to pick the epoch."""
    gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=RANDOM_SEED)
    _, va = next(gss.split(train_df, groups=train_df["patient_id"]))
    return train_df.iloc[va].reset_index(drop=True)

for i, tr, te in folds:
    print(f"fold {i}: val {len(val_split(tr)):,} | test {len(te):,}")
    

fold 0: val 11,209 | test 18,068
fold 1: val 10,912 | test 18,929
fold 2: val 10,643 | test 17,859
fold 3: val 10,894 | test 17,831
fold 4: val 11,263 | test 18,247


## Embed the K pairs with the frozen CNN
`h_k` is the input to the CNN's final `Linear` — the representation its own head used. Cached to disk, so this GPU step runs once.

In [ ]:
@torch.no_grad()
def embed(model, df): # Gets an embeding for each of the query's K pairs of xi,xcf. For every query in df
    """-> h (n, K, d), per-pair probabilities (n, K), labels (n,)."""
    last = model.classifier if isinstance(model.classifier, nn.Linear) else model.classifier[-1] # Finds last layer of the cnn
    grabbed = []
    hook = last.register_forward_pre_hook(lambda m, x: grabbed.append(x[0].float().cpu())) # Adds a hook before the layer, and gets its input. vector h_k
    loader = DataLoader(cfg["dataset"](df, cti.DATA_ROOT, cti.transform),
                        batch_size=EVAL_BATCH, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval() # Inference only, so weights don't change!
    H, P, Y = [], [], []
    try:
        for batch in loader: #batch of querys
            grabbed.clear() 
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                logits, b, k, y = cti._run_batch(model, batch, device, cfg["mode"]) # Run the batch through the model
            H.append(grabbed[0].view(b, k, -1)) # flatten it and save the embeddings for each query's K pairs of xi,xcf. grabbed[0] is the input to the last layer, which is the embedding vector h_k
            P.append(torch.sigmoid(logits.float()).view(b, k).cpu()) # save per pair probabilities
            Y.append(y) # save correctness of the query (1 if correct, 0 if not)
    finally:
        hook.remove()
    return torch.cat(H), torch.cat(P), torch.cat(Y).float()


def cached_embeddings(fold, train_df, test_df):
    path = os.path.join(OUT_DIR, f"fold_{fold}_embeddings.pt")
    if os.path.exists(path):
        return torch.load(path, weights_only=False)
    model = cfg["make_model"]().to(device)
    model.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"fold_{fold}_best.pt"),
                                     map_location=device))
    val_df = val_split(train_df)
    out = {}
    for name, df in (("val", val_df), ("test", test_df)):
        cti._assert_k_alignment(df, K, f"fold {fold} {name}")
        h, p, y = embed(model, df)
        out[name] = dict(h=h, p=p, y=y, pid=df["patient_id"].astype(str).values,
                         path=df["path"].astype(str).values)
    torch.save(out, path)
    return out

## Gated attention head (Eq. 4)
$a_k = \mathrm{softmax}_k\big(\mathbf w^\top[\tanh(\mathbf V\mathbf h_k)\odot\mathrm{sigm}(\mathbf U\mathbf h_k)]\big)$, $\;\mathbf z=\sum_k a_k\mathbf h_k$, then a linear classifier.
`pool="mean"` is the control: same head, uniform weights.

In [9]:
class AttentionHead(nn.Module):
    def __init__(self, d, pool="attn"):
        super().__init__()
        self.pool = pool
        self.norm = nn.LayerNorm(d)
        self.V, self.U = nn.Linear(d, ATTN_DIM), nn.Linear(d, ATTN_DIM)
        self.w = nn.Linear(ATTN_DIM, 1)
        self.cls = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(d, 1))

    def forward(self, h):                                   # h: (B, K, d)
        h = self.norm(h)
        if self.pool == "attn":
            a = torch.softmax(self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h))), dim=1)
        else:
            a = torch.full_like(h[..., :1], 1 / h.shape[1])
        return self.cls((a * h).sum(1)).squeeze(1), a.squeeze(-1)


def predict(head, h):
    head.eval()
    with torch.no_grad():
        logit, a = head(h.to(device))
    return torch.sigmoid(logit).cpu().numpy(), a.cpu().numpy()


def fit_head(d, pool, seed):
    """Fit on val, early-stopped on a patient-disjoint slice of val."""
    torch.manual_seed(seed)
    gss = GroupShuffleSplit(n_splits=1, test_size=HEAD_VAL_FRAC, random_state=seed)
    fit, stop = next(gss.split(d["h"], groups=d["pid"]))
    h, y = d["h"], d["y"]
    head = AttentionHead(h.shape[-1], pool).to(device)
    pos = y[fit].mean().item()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - pos) / pos, device=device))
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best, best_state, stale = -1, None, 0
    for _ in range(EPOCHS):
        head.train()
        for idx in torch.as_tensor(fit)[torch.randperm(len(fit))].split(BATCH):
            logit, _ = head(h[idx].to(device))
            loss = loss_fn(logit, y[idx].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
        auc = roc_auc_score(y[stop].numpy(), predict(head, h[stop])[0])
        if auc > best:
            best, stale = auc, 0
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
        elif (stale := stale + 1) >= PATIENCE:
            break
    head.load_state_dict(best_state)
    return head, best

## Run all folds

In [10]:
rows, weights = [], []
for fold, train_df, test_df in folds:
    d = cached_embeddings(fold, train_df, test_df)
    te = d["test"]
    y = te["y"].numpy()

    row = {"fold": fold, "n_val": len(d["val"]["y"]), "n_test": len(y),
           "mean_sigmoid": roc_auc_score(y, te["p"].mean(1).numpy()),   # current method
           "slot0_only":   roc_auc_score(y, te["p"][:, 0].numpy())}     # = K=1
    for pool in ("mean", "attn"):
        head, head_val = fit_head(d["val"], pool, RANDOM_SEED + fold)
        p, a = predict(head, te["h"])
        row[f"{pool}_pool"] = roc_auc_score(y, p)
        row[f"{pool}_headval"] = head_val
        if pool == "attn":
            weights.append(a)
            pd.DataFrame({"path": te["path"], "correct": y, "attn_prob": p,
                          **{f"a_{k}": a[:, k] for k in range(K)}}
                         ).to_csv(os.path.join(OUT_DIR, f"fold_{fold}_predictions.csv"), index=False)
    rows.append(row)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "fold_aucs.csv"), index=False)

AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Results
`mean_sigmoid` should match your existing `cf_{K}/images` number — if it doesn't, something differs from the K sweep (folds, checkpoint or `VAL_FRAC`).

In [ ]:
cols = ["slot0_only", "mean_sigmoid", "mean_pool", "attn_pool"]
summary = res[cols].agg(["mean", "std"]).T
summary["Δ vs mean_sigmoid"] = res[cols].sub(res["mean_sigmoid"], axis=0).mean()
summary.round(4)

### Where does attention go?
Mean weight per slot (slot 0 = nearest CF), split by whether C0 was correct. Uniform would be 1/K everywhere.

In [ ]:
import matplotlib.pyplot as plt
A = np.concatenate(weights)
y_all = np.concatenate([pd.read_csv(os.path.join(OUT_DIR, f"fold_{f}_predictions.csv"))["correct"]
                        for f in range(N_FOLDS)])
x = np.arange(K)
plt.bar(x - 0.2, A[y_all == 1].mean(0), 0.4, label="C0 correct")
plt.bar(x + 0.2, A[y_all == 0].mean(0), 0.4, label="C0 wrong")
plt.axhline(1 / K, ls="--", c="grey")
plt.xticks(x, [f"CF{k+1}" for k in x]); plt.ylabel("mean attention"); plt.legend(); plt.show()